# Stage 2: Deep Neural Network

This stage replaces the single sigmoid neuron with a fully connected network. Each layer computes `Z[l] = W[l] A[l-1] + b[l]`; hidden layers use ReLU, and the final layer uses sigmoid to produce a binary probability.

The useful jump is representation learning: instead of learning one direct pixel-to-label boundary, the hidden layers learn intermediate features that can make the final boundary easier.

Import NumPy for all model math. The inline comment records the class-label convention used while experimenting with the dataset.


In [ ]:
import numpy as np
#1-29 is gopher


Load the saved dataset. The same `data/gopher_dataset.npz` archive is used, but this stage feeds it into a deeper network.


In [ ]:
loaded_data = np.load("data/gopher_dataset.npz")

Extract the training and test arrays from the dataset archive.


In [ ]:
X_train = loaded_data["X_train"]
y_train = loaded_data["y_train"]
X_test = loaded_data["X_test"]
y_test = loaded_data["y_test"]

Inspect the raw training-image shape before flattening. The first dimension is the number of examples `m`; the remaining dimensions become the input width `n` after flattening.


In [ ]:
X_train.shape

Flatten and normalize the images. A dense layer expects vectors, so each `(height, width, channels)` image becomes one feature vector; dividing by 255 keeps activations and gradients on a friendlier numeric scale.

Labels are reshaped to `(m, 1)` so the sigmoid output and true labels line up in the binary cross-entropy calculation.


In [ ]:
X_flat = X_train.reshape(X_train.shape[0], -1) / 255.0
X_test = X_test.reshape(X_test.shape[0], -1) / 255.0
y_train = y_train.reshape(y_train.shape[0], 1)
y_test = y_test.reshape(y_test.shape[0], 1)
X_flat.shape

Initialize parameters for every layer. Each `W[l]` maps activations from the previous layer into the next layer: `(layer_dims[l], layer_dims[l-1])`.

Small random weights break symmetry so hidden units do not all learn the same feature; zero biases are safe because the random weights already make the neurons different.


In [ ]:
def initialize_parameters(layer_dims):
    
    W = [np.random.randn(layer_dims[i], layer_dims[i-1]) * 0.01 for i in range(1, len(layer_dims))]
    B = [np.zeros((layer_dims[i], 1)) for i in range(1, len(layer_dims))]
    return W, B

Define the network architecture as input width, hidden widths, and output width. The `64 -> 32 -> 1` stack first expands into learned features, then compresses them into one logit for binary classification.


In [ ]:
layer_dims = [X_flat.shape[1], 64, 32, 1]
W, B = initialize_parameters(layer_dims)
W

Define the forward pass for all layers. The recurrence is `A[0] = X.T`, then `Z[l] = W[l] A[l-1] + b[l]`, then `A[l] = activation(Z[l])`.

ReLU, `max(0, z)`, keeps positive evidence and drops negative values, which helps deeper networks avoid the saturation that sigmoid hidden layers can suffer. Caching `A` and `Z` is useful because backprop needs them for the chain rule.


In [ ]:
def forward_pass(X, W, B, layer_dims):
    cache = {}
    A = X.T
    for i in range(len(layer_dims) - 1):
        print(f"Layer {i+1}: W shape {W[i].shape}, B shape {B[i].shape}, A shape {A.shape}")
        Z = np.dot(W[i], A) + B[i]
        if i == len(layer_dims) - 2:  # Output layer
            A = 1 / (1 + np.exp(-Z))  # Sigmoid activation
        else:  # Hidden layers
            A = np.maximum(0, Z)  # ReLU activation
        cache[i] = (A, Z)

    return A, cache


Define one backpropagation and gradient-descent step. Binary cross-entropy with sigmoid again gives the output-layer shortcut `dZ = A - Y`.

For hidden layers, the chain rule sends error backward as `dZ[l] = W[l+1].T dZ[l+1] * 1[Z[l] > 0]`, where the indicator is the ReLU derivative. Then `dW[l] = dZ[l] A[l-1].T / m` and `dB[l] = mean(dZ[l])` tell each layer how to reduce the loss.


In [ ]:
def gradient_descent(X, Y, W, B, layer_dims, learning_rate = 0.01):
    y_pred, cache = forward_pass(X, W, B, layer_dims)
    epsilon = 1e-7
    y_pred_clipped = np.clip(y_pred, epsilon, 1 - epsilon) # Avoid log(0)
    cost = -np.mean(Y.T * np.log(y_pred_clipped) + (1 - Y.T) * np.log(1 - y_pred_clipped))
    gradients = {}
    
    # Output layer gradient
    output_idx = len(layer_dims) - 2
    gradients[f'dz{output_idx}'] = y_pred_clipped - Y.T
    gradients[f'dW{output_idx}'] = (1 / X.shape[0]) * np.dot(gradients[f'dz{output_idx}'], cache[output_idx - 1][0].T)
    gradients[f'dB{output_idx}'] = (1 / X.shape[0]) * np.sum(gradients[f'dz{output_idx}'], axis=1, keepdims=True)
    
    # Hidden layers gradients
    for i in range(output_idx - 1, -1, -1):
        gradients[f'dz{i}'] = np.dot(W[i + 1].T, gradients[f'dz{i + 1}']) * (cache[i][1] > 0)
        if i > 0:
            gradients[f'dW{i}'] = (1 / X.shape[0]) * np.dot(gradients[f'dz{i}'], cache[i - 1][0].T)
        else:
            gradients[f'dW{i}'] = (1 / X.shape[0]) * np.dot(gradients[f'dz{i}'], X)
        gradients[f'dB{i}'] = (1 / X.shape[0]) * np.sum(gradients[f'dz{i}'], axis=1, keepdims=True)
    
    # Update weights and biases
    for i in range(len(layer_dims) - 1):
        W[i] -= learning_rate * gradients[f'dW{i}']
        B[i] -= learning_rate * gradients[f'dB{i}']
    return W, B, cost


Train the deep network for multiple epochs by repeatedly running forward propagation, backpropagation, and parameter updates. The cost trace is the main signal that the network is actually descending the loss surface.


In [ ]:
def train_model(X, Y, layer_dims, learning_rate = 0.01, epoch = 500):
    W, B = initialize_parameters(layer_dims)
    for i in range(epoch):
        W, B, cost = gradient_descent(X, Y, W, B, layer_dims, learning_rate)
        print(f"Cost for epoch {i} is {cost}")
    return W, B


Convert output probabilities into binary labels with a 0.5 decision threshold. This threshold corresponds to a zero final logit, so the model predicts class 1 only when the learned evidence is positive.


In [ ]:
def predict(pred):
    return (pred > 0.5).astype(int)

Train the network, then generate predictions for the test set. Because this model has many more parameters than logistic regression, comparing train and test behavior matters: high train accuracy alone can mean memorization.


In [ ]:
print(X_flat.shape, y_train.shape)
W, B = train_model(X_flat, y_train, layer_dims, epoch=5000)
pred = forward_pass(X_test, W, B, layer_dims)[0]

Compute accuracy and visualize one test prediction from the trained deep network. Accuracy checks the final decisions, while the probability shows how far the output sigmoid was from the 0.5 boundary.


In [ ]:
import matplotlib.pyplot as plt

train_acc = (predict(forward_pass(X_flat, W, B, layer_dims)[0]) == y_train).mean()
test_acc = (predict(forward_pass(X_test, W, B, layer_dims)[0]) == y_test).mean()
print(f"train acc: {train_acc:.3f}   test acc: {test_acc:.3f}")

test_image = X_test[8].reshape(1, -1)
prediction_prob = forward_pass(test_image, W, B, layer_dims)[0]
prediction = predict(prediction_prob)

img_display = (X_test[8].reshape(128, 128, 3) * 255).astype(np.uint8)

plt.figure(figsize=(6, 6))
plt.imshow(img_display)
plt.title(f"Pred: {'Gopher' if prediction[0][0] == 1 else 'Not Gopher'} ({prediction_prob[0][0]:.4f}) | True: {'Gopher' if y_test[0][0] == 1 else 'Not Gopher'}")
plt.axis('off')
plt.show()


Repeat the prediction visualization for the first test image. Looking at individual examples is useful because the network may be right for the wrong visual reason, especially with a small image dataset.


In [ ]:
import matplotlib.pyplot as plt

train_acc = (predict(forward_pass(X_flat, W, B, layer_dims)[0]) == y_train).mean()
test_acc = (predict(forward_pass(X_test, W, B, layer_dims)[0]) == y_test).mean()
print(f"train acc: {train_acc:.3f}   test acc: {test_acc:.3f}")

test_image = X_test[0].reshape(1, -1)
prediction_prob = forward_pass(test_image, W, B, layer_dims)[0]
prediction = predict(prediction_prob)

img_display = (X_test[0].reshape(128, 128, 3) * 255).astype(np.uint8)

plt.figure(figsize=(6, 6))
plt.imshow(img_display)
plt.title(f"Pred: {'Gopher' if prediction[0][0] == 1 else 'Not Gopher'} ({prediction_prob[0][0]:.4f}) | True: {'Gopher' if y_test[0][0] == 1 else 'Not Gopher'}")
plt.axis('off')
plt.show()


Show a grid of test images with true labels so the model inputs can be inspected visually. This helps judge whether mistakes are likely model failures, ambiguous images, or noisy labels.


In [ ]:
import matplotlib.pyplot as plt
import numpy as np

# show a grid of test images with true labels
fig, axes = plt.subplots(2, 6, figsize=(14, 10))
for i, ax in enumerate(axes.flat):
    img = (X_test[i].reshape(128, 128, 3) * 255).astype(np.uint8)
    ax.imshow(img)
    ax.set_title(f"y={int(y_test[i][0])}")
    ax.axis('off')
plt.tight_layout()
plt.show()


Reload and display raw examples from both classes. Side-by-side examples make the classification boundary more concrete than the loss alone.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt

d = np.load("data/gopher_dataset.npz")
Xtr_raw, ytr_raw = d["X_train"], d["y_train"]
Xte_raw, yte_raw = d["X_test"], d["y_test"]
print("shapes:", Xtr_raw.shape, ytr_raw.shape, Xte_raw.shape, yte_raw.shape)

# show 6 gophers, 6 non-gophers
gophers = Xtr_raw[ytr_raw == 1][:6]
nongophers = Xtr_raw[ytr_raw == 0][:6]

fig, axes = plt.subplots(2, 6, figsize=(14, 5))
for i in range(6):
    axes[0, i].imshow(gophers[i]); axes[0, i].axis("off"); axes[0, i].set_title("gopher")
    axes[1, i].imshow(nongophers[i]); axes[1, i].axis("off"); axes[1, i].set_title("not gopher")
plt.tight_layout(); plt.show()
